# Lab 23-05 · An HBM Stack's Heat and an Interposer Link's Energy

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import hbm


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · ngspice

*Predict* the base rises and the transition energy first.

In [2]:
AREA = 12.8e-3 * 11e-3
RL = hbm.layer_resistance(AREA, 30e-6, 150, 10e-6, 1.5)
sim = ngspice("hbm_lab.cir")
for n, key in ((12, "dt12"), (16, "dt16")):
    print(f"{n} layers: model {hbm.base_rise(n, RL, 10, 1):.4f} K, ngspice {sim[key]:.4f} K")
print(f"rising-transition energy: model {1.3e-12 * 0.4**2 * 1e12:.4f} pJ, ngspice {sim['e_rise'] * 1e12:.4f} pJ")

12 layers: model 9.0710 K, ngspice 9.0710 K
16 layers: model 13.6553 K, ngspice 13.6553 K
rising-transition energy: model 0.2080 pJ, ngspice 0.2080 pJ


## Part 2 · Stack height and gap conductivity

*Predict* how the rise grows with layer count.

In [3]:
layers = np.arange(4, 21)
plt.figure()
for label, k_gap, t_gap in (("1.5 W/m-K gaps", 1.5, 10e-6), ("3 W/m-K gaps", 3.0, 10e-6), ("hybrid bonding", 1.0, 0.0)):
    r = hbm.layer_resistance(AREA, 30e-6, 150, t_gap, k_gap)
    plt.plot(layers, [hbm.base_rise(int(n), r, 10, 1) for n in layers], "o-", label=label)
plt.xlabel("layers"); plt.ylabel("base-die rise (K)"); plt.legend(); plt.show()
n_max = max(n for n in range(1, 30) if hbm.base_rise(n, RL, 10, 1) <= 10)
print(f"largest layer count with a rise of at most 10 K: {n_max}")
target = hbm.base_rise(12, RL, 10, 1)
k = optimize.brentq(lambda kg: hbm.base_rise(16, hbm.layer_resistance(AREA, 30e-6, 150, 10e-6, kg), 10, 1) - target, 0.5, 20)
print(f"gap conductivity for 16 layers to match 12: {k:.3f} W/m-K")

largest layer count with a rise of at most 10 K: 12
gap conductivity for 16 layers to match 12: 2.293 W/m-K


## Part 3 · Bandwidth and capacity

*Predict* how many stacks 1 TB at 20 TB/s needs.

In [4]:
rows = [("HBM3 6.4 Gb/s", 1024, 6.4e9), ("HBM3E 9.6 Gb/s", 1024, 9.6e9), ("HBM4 8 Gb/s", 2048, 8e9),
        ("HBM4 11.7 Gb/s", 2048, 11.7e9), ("HBM4E 14 Gb/s", 2048, 14e9)]
for name, w, r in rows:
    print(f"{name:15s}: {hbm.stack_bandwidth(w, r)/1e12:5.2f} TB/s per stack")
print(f"36 GB stacks for 1 TB: {math.ceil(1000/36)}; 2.048 TB/s stacks for 20 TB/s: {math.ceil(20/2.048)}")

HBM3 6.4 Gb/s  :  0.82 TB/s per stack
HBM3E 9.6 Gb/s :  1.23 TB/s per stack
HBM4 8 Gb/s    :  2.05 TB/s per stack
HBM4 11.7 Gb/s :  3.00 TB/s per stack
HBM4E 14 Gb/s  :  3.58 TB/s per stack
36 GB stacks for 1 TB: 28; 2.048 TB/s stacks for 20 TB/s: 10
